In [17]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.


In [ ]:
# Import libraries
import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import roc_curve, auc
from sklearn.preprocessing import label_binarize
from sklearn.metrics import balanced_accuracy_score, f1_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
import seaborn as sns

In [19]:
# Load the cleaned dataset
df = pd.read_csv("../data/processed/cleaned_dataset.csv")

# Quick check
df.head()

,Age,gender,ethnicity,education_level,income_level,employment_status,smoking_status,alcohol_consumption_per_week,physical_activity_minutes_per_week,diet_score,...,hdl_cholesterol,ldl_cholesterol,triglycerides,glucose_fasting,glucose_postprandial,insulin_level,hba1c,diabetes_risk_score,diabetes_stage,diagnosed_diabetes
0,58,male,asian,highschool,Lower-Middle,Employed,Never,0,215,5.7,...,41,160,145,136,236,6.36,8.18,29.6,Type 2,1
2,60,male,hispanic,highschool,Middle,Unemployed,Never,1,57,6.4,...,66,99,36,118,195,5.07,7.51,44.7,Type 2,1
3,74,female,black,highschool,Low,Retired,Never,0,49,3.4,...,50,79,140,139,253,5.28,9.03,38.2,Type 2,1
4,46,male,white,graduate,Middle,Retired,Never,1,109,7.2,...,52,125,160,137,184,12.74,7.20,23.5,Type 2,1
5,46,female,white,highschool,Upper-Middle,Employed,Never,2,124,9.0,...,61,119,179,100,133,8.77,6.03,23.5,Pre-Diabetes,0


In [20]:
# One-hot encode input features
X = pd.get_dummies(df.drop("diabetes_stage", axis=1), drop_first=True)

# Label encode target
le = LabelEncoder()
y = le.fit_transform(df["diabetes_stage"])

In [21]:
# Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [22]:
#SMOTE for class imbalance
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print("Original training set shape:", X_train.shape, "Class distribution:", np.bincount(y_train))
print("Resampled training set shape:", X_train_resampled.shape, "Class distribution:", np.bincount(y_train_resampled))

Original training set shape: (71648, 42) Class distribution: [  214 24810    94 46530]
Resampled training set shape: (186120, 42) Class distribution: [46530 46530 46530 46530]


In [24]:
# Train a Random Forest on the resampled data
rf = RandomForestClassifier(
    n_estimators=300,       # more trees for stability
    max_depth=None,         # allow full depth
    random_state=42,
    n_jobs=-1
)
rf.fit(X_train_resampled, y_train_resampled)

# Evaluate on the original test set
preds = rf.predict(X_test)

print("Balanced Accuracy:", balanced_accuracy_score(y_test, preds))
print("Macro F1:", f1_score(y_test, preds, average='macro'))
print("\nClassification Report:\n", classification_report(y_test, preds, zero_division=0))

# Confusion matrix visualization
cm = confusion_matrix(y_test, preds)
plt.figure(figsize=(6,4))
sns.heatmap(cm, annot=True, fmt="d", cmap="Spectral")
plt.title("Random Forest Confusion Matrix")
plt.xlabel("Predicted label")
plt.ylabel("True label")
plt.show()

NameError: name 'balanced_accuracy_score' is not defined